In [123]:
import sys
import os
import urllib3
from configparser import ConfigParser

# Add your local ThreatConnect SDK to path
sys.path.append(r"Z:\HTOC\Data_Analytics\threatconnect")
from ThreatConnect import ThreatConnect
from RequestObject import RequestObject
from Owners import Owners

# Add your project repo to path
project_root = r"C:\Users\jaskew\Documents\project_repository\scripts\Data Movement\ThrearConnect-api-pull"
if project_root not in sys.path:
    sys.path.append(project_root)

from utils.config_loader import load_config

# Load API config
config_path = os.path.join(project_root, "utils", "config.json")
try:
    api_secret_key, api_access_id, api_base_url, api_default_org = load_config(config_path)
    display(f"Loaded config from: {config_path}")
    display(f"Base URL: {api_base_url}")
    display(f"Access ID: {api_access_id}")
    display(f"Default Org: {api_default_org}")
except Exception as e:
    display(f"[ERROR] Failed to load configuration: {e}")
    sys.exit(1)

# Disable SSL verification warnings (use cautiously)
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
verify_ssl = False

# Initialize ThreatConnect session
try:
    tc = ThreatConnect(api_access_id, api_secret_key, api_default_org, api_base_url)
    display("ThreatConnect initialized.")
except Exception as e:
    display(f"[ERROR] Failed to initialize ThreatConnect: {e}")
    sys.exit(1)

# Define the owner (organization scope)
owner = 'HTOC Org'

# Create a request object to fetch indicators (or other data)
try:
    ro = RequestObject()
    ro.set_http_method('GET')
    ro.set_owner(owner)
    ro.set_owner_allowed(True)
    # ro.set_resource_pagination(True)  # Uncomment if needed
    display("RequestObject successfully created.")
except Exception as e:
    display(f"[ERROR] Failed to initialize RequestObject: {e}")
    sys.exit(1)




'Loaded config from: C:\\Users\\jaskew\\Documents\\project_repository\\scripts\\Data Movement\\ThrearConnect-api-pull\\utils\\config.json'

'Base URL: https://hvs.threatconnect.com/api'

'Access ID: 09783848890162390382'

'Default Org: HTOC Org'

'ThreatConnect initialized.'

'RequestObject successfully created.'

In [124]:
import pandas as pd
from datetime import datetime, timedelta
import pytz
import urllib.parse

# Setup
cutoff = pd.Timestamp.utcnow()
start_date = (datetime.now(pytz.UTC) - timedelta(days=3)).date()
start = f"{start_date}T00:00:00Z"
type_names = ["Address", "EmailAddress", "File", "Host", "URL", "ASN", "CIDR",
              "Email Subject", "Hashtag", "Mutex", "Registry Key", "Stripped URL", "User Agent"]
type_name_condition = ", ".join([f'"{t}"' for t in type_names])
list_of_owners = ['HTOC Org']
final_results = []

# Query indicators
for owner in list_of_owners:
    display(f"Querying owner: {owner}")
    try:
        tql_raw = (
            f'ownerName EQ "{owner}" AND '
            f'typeName IN ({type_name_condition}) AND '
            f'lastObserved >= "{start}"'
        )
        tql_encoded = urllib.parse.quote(tql_raw)
        ro.set_http_method('GET')
        ro.set_request_uri(f'/v3/indicators?tql={tql_encoded}&fields=tags,observations,associatedGroups,falsePositives,&resultStart=0&resultLimit=10000')
        response = tc.api_request(ro)

        if response.headers.get('content-type') == 'application/json':
            results = response.json()
            final_results.append(results)
    except Exception as e:
        display(f"Failed to query indicators for {owner}: {e}")

# Normalize results
normalized_data = []
for result in final_results:
    data_items = result.get('data', [])
    if not data_items:
        display("No data returned in API response:", result)
    for item in data_items:
        if isinstance(item, dict) and 'summary' in item:
            normalized_data.append(item)

if normalized_data:
    observed_src = pd.json_normalize(normalized_data)
    observed_src['indicator'] = observed_src['summary'].str.split().str[0].str.strip()
    observed_src.drop_duplicates(subset='indicator', inplace=True)
    observed_src['lastObserved'] = pd.to_datetime(observed_src['lastObserved'], utc=True)
    observed_src = observed_src[observed_src['lastObserved'] >= pd.to_datetime(start)]
else:
    display("No valid indicator data found.")
    observed_src = pd.DataFrame()

display(observed_src)


'Querying owner: HTOC Org'

,id,dateAdded,ownerId,ownerName,webLink,type,lastModified,rating,confidence,description,...,tags.data,associatedGroups.data,source,hostName,dnsActive,whoisActive,address,url,lastFalsePositive,indicator
0,6755399459033744,2025-06-16T17:42:31Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-11-10T12:15:20Z,3.0,59,RITM0589984,...,"[{'id': 35760, 'name': 'OS Splunk API', 'descr...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,185.40.4.92
1,6755399465229552,2025-07-28T17:34:31Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-11-10T11:35:07Z,3.0,65,TASK0902923,...,"[{'id': 471298, 'name': 'DHA Splunk API', 'las...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,64.62.156.112
2,6755399465229544,2025-07-28T17:34:26Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-11-10T11:35:07Z,3.0,65,TASK0902923,...,"[{'id': 471298, 'name': 'DHA Splunk API', 'las...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,64.62.156.168
3,6755399465229542,2025-07-28T17:34:22Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-11-10T11:35:07Z,3.0,65,TASK0902923,...,"[{'id': 471298, 'name': 'DHA Splunk API', 'las...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,64.62.156.210
4,6755399465229538,2025-07-28T17:34:18Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-11-10T11:35:07Z,3.0,65,TASK0902923,...,"[{'id': 471298, 'name': 'DHA Splunk API', 'las...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,65.49.1.166
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
836,5271618,2025-01-29T16:27:21Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Stripped URL,2025-02-01T23:22:46Z,3.0,100,The following list of urls and domains are cur...,...,"[{'id': 461545, 'name': 'DeepSeek', 'lastUsed'...","[{'id': 548570, 'dateAdded': '2025-01-29T16:27...",https://cdn.deepseek.com,NaN,NaN,NaN,NaN,cdn.deepseek.com,NaN,cdn.deepseek.com
837,5265377,2025-01-23T20:39:30Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Stripped URL,2025-02-01T23:22:44Z,5.0,91,Gootloader related indicators pulled from VT.,...,"[{'id': 35057, 'name': 'FDA Splunk API', 'last...","[{'id': 546895, 'dateAdded': '2025-01-23T20:38...",https://fitnessenhancement.com,NaN,NaN,NaN,NaN,fitnessenhancement.com,NaN,fitnessenhancement.com
838,4883044,2024-09-09T11:22:22Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Stripped URL,2025-01-25T23:24:38Z,3.0,90,ACD R&F processed a malspam campaign with a Ne...,...,"[{'id': 390199, 'name': 'hhs-2024090901', 'las...","[{'id': 455233, 'dateAdded': '2024-09-09T11:22...",https://www.shorturl.at/,NaN,NaN,NaN,NaN,www.shorturl.at/,NaN,www.shorturl.at/
839,4476331,2023-11-16T13:43:29Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Stripped URL,2025-01-17T23:24:44Z,4.0,85,NaN,...,"[{'id': 35760, 'name': 'OS Splunk API', 'descr...","[{'id': 291847, 'dateAdded': '2023-12-15T13:16...",http://selligenttier.naylorcampaigns.com,NaN,NaN,NaN,NaN,selligenttier.naylorcampaigns.com,NaN,selligenttier.naylorcampaigns.com


In [125]:
# Unnest tags.data in observed_src to get a list of each tag per indicator

# Explode tags.data to one row per tag
tags_exploded = (
    observed_src[['indicator', 'tags.data']]
      .explode('tags.data')
      .dropna(subset=['tags.data'])
)

# Extract tag name from each tag object
tags_exploded['tag_name'] = tags_exploded['tags.data'].apply(lambda x: x.get('name') if isinstance(x, dict) else None)

# Aggregate all tag names into a list per indicator
tags_per_indicator = (
    tags_exploded.groupby('indicator')['tag_name']
    .apply(lambda x: [t for t in x if t])
    .reset_index()
    .rename(columns={'tag_name': 'tag_list'})
)

# Merge back to observed_src if desired
observed_src_with_tags = observed_src.merge(tags_per_indicator, on='indicator', how='left')


In [126]:
observed_src_with_tags[observed_src_with_tags['indicator'] == '192.42.116.192']

,id,dateAdded,ownerId,ownerName,webLink,type,lastModified,rating,confidence,description,...,associatedGroups.data,source,hostName,dnsActive,whoisActive,address,url,lastFalsePositive,indicator,tag_list
349,4501750,2024-01-18T14:44:39Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-11-09T13:22:49Z,3.0,60,RITM0589984,...,"[{'id': 309137, 'dateAdded': '2024-01-24T20:33...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,192.42.116.192,"[Micorosoft login Impersonation, TOR Activity,..."


In [127]:
observed_src_with_tags[observed_src_with_tags['tag_list'].apply(lambda tags: isinstance(tags, list) and any(isinstance(tag, str) and 'Scan' in tag for tag in tags))]

,id,dateAdded,ownerId,ownerName,webLink,type,lastModified,rating,confidence,description,...,associatedGroups.data,source,hostName,dnsActive,whoisActive,address,url,lastFalsePositive,indicator,tag_list
48,5629499547737367,2025-06-04T16:43:19Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-11-10T11:16:35Z,3.0,59,TASK0883886 / RITM0586633,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,193.37.69.113,"[Web-Scanner, DHA Splunk API, OS Splunk API, F..."
115,6755399443015046,2025-03-14T11:55:19Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-11-10T08:41:01Z,4.0,45,CMS received a Volumetrics alert regarding mul...,...,"[{'id': 6755399443002242, 'dateAdded': '2025-0...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,91.196.152.71,"[DHA Splunk API, Active Scanning: Scanning IP ..."
272,5629499555060719,2025-06-16T17:42:49Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-11-10T07:16:51Z,4.0,62,TASK0890150,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,91.191.209.74,"[Bulgaria, DHA Splunk API, Web Scanner, OS Spl..."
273,4554082,2024-03-29T14:31:35Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-11-10T07:16:07Z,1.0,30,CMS Anomali ThreatStream Indicators from 02.14...,...,"[{'id': 327568, 'dateAdded': '2024-03-29T14:31...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,118.193.72.187,"[DHA Splunk API, Web Scanner, cms-soc, OS Splu..."
277,5629499536006284,2025-03-10T13:55:27Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-11-10T06:36:26Z,2.0,53,The following IP that made multiple attempts (...,...,"[{'id': 6755399443001263, 'dateAdded': '2025-0...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,59.24.28.114,"[DHA Splunk API, Active Scanning, OS Splunk AP..."
292,6755399443015052,2025-03-14T11:55:20Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-11-10T04:17:30Z,4.0,46,CMS received a Volumetrics alert regarding mul...,...,"[{'id': 6755399443002242, 'dateAdded': '2025-0...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,91.196.152.67,"[DHA Splunk API, Active Scanning: Scanning IP ..."
293,6755399443015051,2025-03-14T11:55:19Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-11-10T04:17:30Z,4.0,48,CMS received a Volumetrics alert regarding mul...,...,"[{'id': 6755399443002242, 'dateAdded': '2025-0...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,91.196.152.56,"[DHA Splunk API, Active Scanning: Scanning IP ..."
294,6755399443015049,2025-03-14T11:55:19Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-11-10T04:17:30Z,4.0,47,CMS received a Volumetrics alert regarding mul...,...,"[{'id': 6755399443002242, 'dateAdded': '2025-0...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,91.196.152.44,"[DHA Splunk API, Active Scanning: Scanning IP ..."
295,6755399443015048,2025-03-14T11:55:19Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-11-10T04:17:30Z,4.0,46,CMS received a Volumetrics alert regarding mul...,...,"[{'id': 6755399443002242, 'dateAdded': '2025-0...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,91.196.152.66,"[DHA Splunk API, Active Scanning: Scanning IP ..."
296,6755399443015047,2025-03-14T11:55:19Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-11-10T04:17:30Z,4.0,47,CMS received a Volumetrics alert regarding mul...,...,"[{'id': 6755399443002242, 'dateAdded': '2025-0...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,91.196.152.46,"[DHA Splunk API, Active Scanning: Scanning IP ..."


In [128]:
# List of tags to count (case-insensitive, strip whitespace)
tags_of_interest = [
    "Scanning", "DDoS", "Spam", "Phishing", "Cryptojacking",
    "Credential Stuffing", "Ransomware", "Data Theft",
    "Cross Site Scripting Attacks", "SQL Injections"
]

# Normalize tag_list to lowercase for matching
def tag_counter(tag_list, tag):
    if not isinstance(tag_list, list):
        return 0
    return sum(1 for t in tag_list if isinstance(t, str) and t.strip().lower() == tag.lower())

tag_counts = {}
for tag in tags_of_interest:
    tag_counts[tag] = observed_src_with_tags['tag_list'].apply(lambda lst: tag_counter(lst, tag)).sum()
    # Add a column to observed_src with the list of matching "Botnet" tags per indicator
    botnet_tags = set(tags_of_interest)
    def extract_botnet_tags(tag_list):
        if not isinstance(tag_list, list):
            return []
        return [t for t in tag_list if isinstance(t, str) and t.strip().lower() in {b.lower() for b in botnet_tags}]

    # Map indicator to its tag_list for efficient lookup
    indicator_to_tags = dict(zip(observed_src_with_tags['indicator'], observed_src_with_tags['tag_list']))

    observed_src['Botnet'] = observed_src['indicator'].map(lambda ind: extract_botnet_tags(indicator_to_tags.get(ind, [])))
# Display as a DataFrame for readability
pd.DataFrame(list(tag_counts.items()), columns=['Tag', 'Count'])

,Tag,Count
0,Scanning,0
1,DDoS,45
2,Spam,0
3,Phishing,12
4,Cryptojacking,0
5,Credential Stuffing,0
6,Ransomware,4
7,Data Theft,10
8,Cross Site Scripting Attacks,0
9,SQL Injections,0


In [129]:
import os
import pandas as pd
from datetime import datetime, timedelta

# Base file path with placeholder for date
base_path = r"Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d{date}.csv"
#base_path = r"C:\Users\jaskew\Documents\project_repository\data\raw\ObservationDataFiles\htoc_opdiv_obs_d{date}.csv"
date_format = "%Y%m%d"

def get_file_paths(base_path, days=3):
    """ Generate file paths for the last `days` days using list comprehension. """
    today = datetime.utcnow()
    dates_to_pull = [(today - timedelta(days=i)).strftime(date_format) for i in range(days)]
    
    # Construct file paths
    file_paths = [base_path.format(date=dt) for dt in dates_to_pull]
    
    # Filter for existing files
    existing_files = [file_path for file_path in file_paths if os.path.exists(file_path)]
    
    if not existing_files:
        display("No files found for the specified date range.")
    else:
        display(f"Files to be loaded: {existing_files}")
    
    return existing_files

def load_observed_data(file_paths):
    """ Load and concatenate observed data from multiple files. """
    data_frames = []

    for file_path in file_paths:
        try:
            df = pd.read_csv(file_path)
            data_frames.append(df)
        except Exception as e:
            display(f"Error reading file {file_path}: {e}")
    
    # Concatenate data
    if data_frames:
        observed_data_df = pd.concat(data_frames, ignore_index=True)
        display(f"Loaded data from {len(data_frames)} files.")
    else:
        observed_data_df = pd.DataFrame()

    return observed_data_df

# Example Usage:
# Fetch file paths for the last 3 days
file_paths = get_file_paths(base_path, days=5)

# Load observed data
observed_data_df = load_observed_data(file_paths)



C:\Users\jaskew\AppData\Local\Temp\ipykernel_19924\4102358923.py:12: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  today = datetime.utcnow()


"Files to be loaded: ['Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20251110.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20251109.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20251108.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20251107.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20251106.csv']"

'Loaded data from 5 files.'

In [130]:
import pandas as pd

df = observed_src.copy()

# explode tags.data
tags_exploded = (
    df[['indicator', 'tags.data']]
      .explode('tags.data')
      .dropna(subset=['tags.data'])
)

# normalize all fields in tags.data into flat columns
tags_norm = pd.json_normalize(tags_exploded['tags.data'])
tags_norm.columns = [f"tag_{c}" for c in tags_norm.columns]

# re‑attach indicator
tags_expanded = tags_exploded.reset_index(drop=True).join(tags_norm)

# extract partners
tags_expanded['partner'] = tags_expanded['tag_name'].map(
    lambda n: n[:-len(' Splunk API')] if isinstance(n, str) and n.endswith(' Splunk API') else None
)

# aggregate each tag_* field into a list per indicator
tag_fields = list(tags_norm.columns)
tag_agg = (
    tags_expanded
      .groupby('indicator', as_index=False)
      .agg({
          **{f: list for f in tag_fields},
          'partner': lambda x: [p for p in dict.fromkeys(x) if p]
      })
      .rename(columns={'partner':'partners'})
)


# --- CORE AGGREGATION OF OTHER COLUMNS ---

first_cols = [
    'id','dateAdded','ownerId','ownerName','webLink','type','lastModified', 'falsePositives',
    'rating','confidence','description','summary','observations',
    'lastObserved','privateFlag','active','activeLocked','ip',
    'legacyLink','source','address','url'
]
# Remove 'hostName', 'dnsActive', 'whoisActive' from first_cols to avoid KeyError

# Add 'Botnet' column from observed_src if it exists
if 'Botnet' in observed_src.columns:
    df['Botnet'] = observed_src['Botnet']
    first_cols.append('Botnet')

base_agg = (
    df
      .drop(columns=[
          'createdBy.id','createdBy.userName','createdBy.firstName','createdBy.lastName',
          'createdBy.pseudonym','createdBy.owner','xid','eventType','documentDateAdded',
          'documentType','fileSize','fileName','downVoteCount','upVoteCount','type_group',
          'webLink_group','ownerName_group','ownerId_group','dateAdded_group','id_group',
          'platforms.count','tactics.count',
      ], errors='ignore')
      .groupby('indicator', as_index=False)[
          [c for c in first_cols if c in df.columns]
      ]
      .first()
)

# --- MERGE EVERYTHING BACK ---

agg_df = (
    base_agg
      .merge(tag_agg,   on='indicator', how='left')
#      .merge(group_agg, on='indicator', how='left')
)

def clean_list(lst):
    if not isinstance(lst, list):
        return []
    cleaned = []
    for v in lst:
        # drop any null‑like values
        try:
            if pd.isna(v):
                continue
        except Exception:
            pass
        # drop empty strings
        if isinstance(v, str) and v == "":
            continue
        cleaned.append(v)
    return cleaned

def list_to_csv(lst):
    """
    Takes a cleaned list and returns:
      - a comma-separated string of its items, OR
      - an empty string if there are none.
    """
    if not lst:
        return ""
    return ", ".join(str(v) for v in lst)


for col in ['partners', 'group_ids', 'group_names'] + tag_fields:
    if col in agg_df.columns:
        agg_df[col] = agg_df[col].apply(clean_list).apply(list_to_csv)
    
display(agg_df)



,indicator,id,dateAdded,ownerId,ownerName,webLink,type,lastModified,falsePositives,rating,...,tag_id,tag_name,tag_description,tag_lastUsed,tag_techniqueId,tag_tactics.data,tag_tactics.count,tag_platforms.data,tag_platforms.count,partners
0,101.71.130.99,5629499572136493,2025-10-16T15:41:04Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-11-09T13:24:45Z,0,3.0,...,"471298, 35760, 35057, 30479, 23630, 23577, 235...","DHA Splunk API, OS Splunk API, FDA Splunk API,...",Observations reported by the HHS Ofice of the ...,"2025-11-10T00:10:39Z, 2025-11-06T19:12:02Z, 20...",,,,,,"DHA, OS, FDA, CMS, NIH, HHS"
1,101.89.174.236,5629499542017370,2025-05-21T18:39:43Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-11-10T11:16:35Z,0,3.0,...,"471298, 35760, 35057, 30479, 23667, 23630, 235...","DHA Splunk API, OS Splunk API, FDA Splunk API,...",Observations reported by the HHS Ofice of the ...,"2025-11-10T00:10:39Z, 2025-11-06T19:12:02Z, 20...",,,,,,"DHA, OS, FDA, CMS, HRSA, NIH, IHS, HHS"
2,103.133.60.12,6755399460008266,2025-07-02T12:05:36Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-11-07T12:19:46Z,0,1.0,...,"1469320, 1455870, 505200, 471298, 35057, 23769...","INDICATOR NOTICE 25178.1, Mr Hamza Group, T-Su...",,"2025-07-18T13:37:52Z, 2025-10-21T17:48:35Z, 20...",,,,,,"DHA, FDA, HRSA, HHS"
3,103.150.218.78,6755399460008269,2025-07-02T12:05:36Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-11-08T12:40:06Z,0,2.0,...,"1469320, 1455870, 505200, 471298, 23769, 23667...","INDICATOR NOTICE 25178.1, Mr Hamza Group, T-Su...",,"2025-07-18T13:37:52Z, 2025-10-21T17:48:35Z, 20...",,,,,,"DHA, HRSA, NIH"
4,103.153.149.26,6755399460007963,2025-07-02T12:05:33Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-11-07T12:20:01Z,0,1.0,...,"1469320, 1455870, 505200, 471298, 35057, 23769...","INDICATOR NOTICE 25178.1, Mr Hamza Group, T-Su...",,"2025-07-18T13:37:52Z, 2025-10-21T17:48:35Z, 20...",,,,,,"DHA, FDA, NIH"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
834,www.deepseek.com,5271608,2025-01-29T16:27:10Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Host,2025-11-04T13:20:17Z,0,3.0,...,"471298, 461545, 35760, 35752, 35057, 30770, 30...","DHA Splunk API, DeepSeek, OS Splunk API, VA OI...",Observations reported by the HHS Ofice of the ...,"2025-11-10T00:10:39Z, 2025-01-31T14:04:11Z, 20...",,,,,,"DHA, OS, FDA, CMS, HRSA, NIH, IHS"
835,www.deepseek.com.cdn.cloudflare.net,5271612,2025-01-29T16:27:10Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Host,2025-11-04T13:15:43Z,0,3.0,...,"471298, 461545, 35752, 30479, 23769, 23630, 23...","DHA Splunk API, DeepSeek, VA OIS CSOC CTS Splu...",,"2025-11-10T00:10:39Z, 2025-01-31T14:04:11Z, 20...",,,,,,"DHA, CMS, NIH, IHS"
836,www.shorturl.at/,4883044,2024-09-09T11:22:22Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Stripped URL,2025-01-25T23:24:38Z,0,3.0,...,"390199, 35057, 23576, 23575, 1502, 26","hhs-2024090901, FDA Splunk API, Observed, CDC ...",,"2024-09-10T10:48:34Z, 2025-11-06T22:07:55Z, 20...",,,,,,"FDA, CDC"
837,www.sthda.com,4565837,2024-04-16T17:33:15Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Host,2025-11-06T04:55:46Z,0,4.0,...,"471298, 35760, 35752, 35057, 30770, 30479, 237...","DHA Splunk API, OS Splunk API, VA OIS CSOC CTS...",Observations reported by the HHS Ofice of the ...,"2025-11-10T00:10:39Z, 2025-11-06T19:12:02Z, 20...",,,,,,"DHA, OS, FDA, CMS, HRSA, NIH, IHS"


In [131]:
# ──────────────────────────────────────────────────────────────────────────────
# Clean enrichment: filter unsupported types, call providers separately, summarize
# ──────────────────────────────────────────────────────────────────────────────
import urllib.parse
import pandas as pd

COL_PATH = "data.enrichment.data"   # adjust if API changes

# Determine candidate indicators (use 'indicator' if present, else 'summary')
key_col = 'indicator' if 'indicator' in agg_df.columns else 'summary'

# Keep only types that Shodan/VT commonly support; others will be VT-only
supported_types = {
    'Address','IPv4','IPv6','Host','Domain','URL','File','SHA1','SHA256','MD5'
}

candidates = (
    agg_df[[key_col, 'type']]
      .dropna(subset=[key_col])
      .astype({key_col: str})
      .drop_duplicates(subset=[key_col])
)

indicator_values = candidates[key_col].tolist()

display(f"Enriching {len(indicator_values)} indicators (VT; Shodan where supported)...")

enriched_results = []
failed = []

for _, row in candidates.iterrows():
    value = row[key_col]
    typ   = str(row.get('type', '') or '')

    try:
        encoded_value = urllib.parse.quote(value, safe="")

        # Build provider query: always VT; add Shodan only for supported types
        providers = ["VirusTotalV3"]
        if typ in supported_types and typ not in {"URL","File","SHA1","SHA256","MD5"}:
            # Prefer Shodan for network-y types
            providers.append("Shodan")

        q = urllib.parse.urlencode({"type": providers}, doseq=True)
        enrich_url = f"/v3/indicators/{encoded_value}/enrich?{q}"

        ro = RequestObject()
        ro.set_http_method("POST")
        ro.set_request_uri(enrich_url)
        ro.set_body({})

        resp = tc.api_request(ro)
        # Many provider errors come back with 200 + status in JSON; don't raise on status
        try:
            data = resp.json()
        except Exception:
            data = {"status": getattr(resp, 'status_code', 'n/a'), "raw": getattr(resp, 'text', None)}

        # Attach key for merge
        data[key_col] = value
        enriched_results.append(data)

    except Exception as e:
        failed.append({key_col: value, "type": typ, "error": str(e)})

# If nothing enriched, carry on with base
if not enriched_results:
    display("No enrichment data retrieved.")
    recent_tags = agg_df.copy()
else:
    df_enriched = (
        pd.json_normalize(enriched_results)
          .drop_duplicates(subset=[key_col], keep="last")
    )

    # Merge enrichment block into base
    recent_tags = agg_df.merge(df_enriched, on=key_col, how="left")

    # ── Flatten enrichment payload without creating duplicate base rows ───────
    if COL_PATH in recent_tags.columns:
        exploded = (
            recent_tags[[key_col, COL_PATH]]
            .explode(COL_PATH)
            .dropna(subset=[COL_PATH])
        )

        enrich_flat = pd.json_normalize(exploded[COL_PATH]).add_prefix("enrich_")
        enrich_flat[key_col] = exploded[key_col].values

        # ---- Aggregation helpers ---------------------------------------------
        def _flatten_lists(values):
            out = []
            for v in values:
                if isinstance(v, list):
                    out.extend(v)
                else:
                    out.append(v)
            return out

        def _agg_obj(series: pd.Series):
            vals = [v for v in series.dropna()]
            if not vals:
                return None
            flat = _flatten_lists(vals)
            if all(not isinstance(v, (list, dict)) for v in flat):
                return list(pd.unique(flat))
            return flat

        obj_cols = enrich_flat.select_dtypes("object").columns.difference([key_col])
        num_cols = enrich_flat.columns.difference(obj_cols.union({key_col}))

        agg_dict = {c: _agg_obj for c in obj_cols}
        agg_dict.update({c: "max" for c in num_cols})

        enrich_wide = (
            enrich_flat
              .groupby(key_col, as_index=False)
              .agg(agg_dict)
        )

        recent_tags = (
            recent_tags.drop(columns=[COL_PATH], errors="ignore")
                       .drop_duplicates(subset=[key_col])
                       .merge(enrich_wide, on=key_col, how="left")
        )

    # Minimal success message
    display(f"Enrichment complete for {recent_tags[key_col].notna().sum()} indicators.")

# Compact failure summary without flooding output
if failed:
    fail_df = pd.DataFrame(failed)
    display(f"{len(failed)} indicators failed enrichment (showing up to 10):")
    display(fail_df.head(10))

# Show preview of key enrichment columns if present
preview_cols = [c for c in [key_col, 'enrich_hostNames', 'enrich_domains', 'enrich_tags', 'enrich_vtMaliciousCount'] if c in recent_tags.columns]

recent_tags.drop(columns=['tag_id', 'tag_lastUsed', 'tag_lastModified', 'tag_ownerId', 
                          'tag_ownerName', 'tag_dateAdded', 'tag_description','tag_tactics.count', 
                          'tag_platform.data', 'tag_platform.count', 'data.id', 'data.dateAdded', 'data.ownerId',
                          'data.webLink', 'data.ownerName', 'data.lastModified', 'data.summary', 'data.ip',
                          'data.legacyLink','data.source', 'enrich_cloudProvider', 'enrich_cloudRegion', 'enrich_type',
                          'id'], inplace=True, errors='ignore')
                          
display(recent_tags[preview_cols].head(20))


'Enriching 839 indicators (VT; Shodan where supported)...'

Status Code: 400
Failed API Response: b'{"errCode":"0x1001","message":"The EmailAddress alyssa.palmer@southsidechs.org cannot be enriched with VirusTotal because the indicator type isn\'t supported.","status":"Error"}'
Status Code: 400
Failed API Response: b'{"errCode":"0x1001","message":"The Host api.deepseek.com cannot be enriched with Shodan because the indicator type isn\'t supported.","status":"Error"}'
Status Code: 400
Failed API Response: b'{"errCode":"0x1001","message":"The Host arbourvalley.com cannot be enriched with Shodan because the indicator type isn\'t supported.","status":"Error"}'
Status Code: 400
Failed API Response: b'{"errCode":"0x1001","message":"The Host arpdcresources.ca cannot be enriched with Shodan because the indicator type isn\'t supported.","status":"Error"}'
Status Code: 400
Failed API Response: b'{"errCode":"0x1001","message":"The Stripped URL atlasofthefuture.org cannot be enriched with VirusTotal because the indicator type isn\'t supported.","status":"E

'Enrichment complete for 839 indicators.'

'63 indicators failed enrichment (showing up to 10):'

,indicator,type,error
0,alyssa.palmer@southsidechs.org,EmailAddress,"b'{""errCode"":""0x1001"",""message"":""The EmailAddr..."
1,api.deepseek.com,Host,"b'{""errCode"":""0x1001"",""message"":""The Host api...."
2,arbourvalley.com,Host,"b'{""errCode"":""0x1001"",""message"":""The Host arbo..."
3,arpdcresources.ca,Host,"b'{""errCode"":""0x1001"",""message"":""The Host arpd..."
4,atlasofthefuture.org,Stripped URL,"b'{""errCode"":""0x1001"",""message"":""The Stripped ..."
5,autonotification@concursolutions.com,EmailAddress,"b'{""errCode"":""0x1001"",""message"":""The EmailAddr..."
6,b.pdf-fast.com,Host,"b'{""errCode"":""0x1001"",""message"":""The Host b.pd..."
7,beatthewonderlic.com,Stripped URL,"b'{""errCode"":""0x1001"",""message"":""The Stripped ..."
8,beholdisrael.org,Stripped URL,"b'{""errCode"":""0x1001"",""message"":""The Stripped ..."
9,bimbinlombardia.com,Host,"b'{""errCode"":""0x1001"",""message"":""The Host bimb..."


,indicator,enrich_hostNames,enrich_domains,enrich_tags,enrich_vtMaliciousCount
0,101.71.130.99,None,None,None,10.0
1,101.89.174.236,None,None,[database],9.0
2,103.133.60.12,None,None,None,1.0
3,103.150.218.78,[PROXINET-INTERNET-103-150-218-78.proxinet.co.id],[proxinet.co.id],[vpn],1.0
4,103.153.149.26,None,None,None,0.0
5,103.163.80.62,None,None,[vpn],0.0
6,103.167.91.247,None,None,None,6.0
7,103.176.96.138,[ip.138-96.g-fiber.co.id],[g-fiber.co.id],None,2.0
8,103.191.196.155,[host-196-155.palindo.id],[palindo.id],None,1.0
9,103.203.59.0,[scan-59-0.security.ipip.net],[ipip.net],[scanner],4.0


In [132]:
recent_tags[recent_tags['indicator'] == '190.92.174.36']

,indicator,dateAdded,ownerId,ownerName,webLink,type,lastModified,falsePositives,rating,confidence,...,enrich_city,enrich_country,enrich_domains,enrich_hostNames,enrich_isp,enrich_openPorts,enrich_org,enrich_tags,enrich_vulnerabilities,enrich_vtMaliciousCount
161,190.92.174.36,2025-04-11T17:46:48Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-11-10T08:40:56Z,0,4.0,63,...,None,None,None,None,None,None,None,None,None,1.0


In [133]:
#recent_tags = recent_tags.head(50)
recent_tags.loc[recent_tags['enrich_vtMaliciousCount'].isnull(), 'enrich_vtMaliciousCount'] = 0
display(recent_tags)

,indicator,dateAdded,ownerId,ownerName,webLink,type,lastModified,falsePositives,rating,confidence,...,enrich_city,enrich_country,enrich_domains,enrich_hostNames,enrich_isp,enrich_openPorts,enrich_org,enrich_tags,enrich_vulnerabilities,enrich_vtMaliciousCount
0,101.71.130.99,2025-10-16T15:41:04Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-11-09T13:24:45Z,0,3.0,77,...,None,None,None,None,None,None,None,None,None,10.0
1,101.89.174.236,2025-05-21T18:39:43Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-11-10T11:16:35Z,0,3.0,57,...,[Shanghai],[China],None,None,[China Telecom (Group)],"[{'transport': 'tcp', 'port': 22, 'data': 'SSH...",[CHINANET SHANGHAI PROVINCE NETWORK],[database],None,9.0
2,103.133.60.12,2025-07-02T12:05:36Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-11-07T12:19:46Z,0,1.0,56,...,[Terbanggi Besar],[Indonesia],None,None,[PT Tunas Link Indonesia],"[{'transport': 'tcp', 'port': 53, 'data': ' Re...",[PT Tunas Link Indonesia],None,None,1.0
3,103.150.218.78,2025-07-02T12:05:36Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-11-08T12:40:06Z,0,2.0,56,...,[Batam],[Indonesia],[proxinet.co.id],[PROXINET-INTERNET-103-150-218-78.proxinet.co.id],[PT Proxi Jaringan Nusantara],"[{'transport': 'tcp', 'port': 53}, {'transport...",[PT Proxi Jaringan Nusantara],[vpn],None,1.0
4,103.153.149.26,2025-07-02T12:05:33Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-11-07T12:20:01Z,0,1.0,56,...,[Weleri],[Indonesia],None,None,[PT JARINGANKU SARANA NUSANTARA],"[{'transport': 'tcp', 'port': 2000, 'product':...",[PT Jaringan Sarana Nusantara],None,None,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
834,www.deepseek.com,2025-01-29T16:27:10Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Host,2025-11-04T13:20:17Z,0,3.0,70,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
835,www.deepseek.com.cdn.cloudflare.net,2025-01-29T16:27:10Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Host,2025-11-04T13:15:43Z,0,3.0,68,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
836,www.shorturl.at/,2024-09-09T11:22:22Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Stripped URL,2025-01-25T23:24:38Z,0,3.0,90,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
837,www.sthda.com,2024-04-16T17:33:15Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Host,2025-11-06T04:55:46Z,0,4.0,36,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0


In [134]:
# Count how many indicators are associated with a unique enrich_domains value

# Only proceed if 'enrich_domains' exists in exploded DataFrame
if 'enrich_domains' in exploded.columns:
    # Drop rows where enrich_domains is missing or NaN
    domains_df = exploded.dropna(subset=['enrich_domains'])

    # If enrich_domains is a list, flatten to individual domain rows
    def flatten_domains(row):
        val = row['enrich_domains']
        if isinstance(val, list):
            return [(row['indicator'], d) for d in val if pd.notna(d)]
        elif pd.notna(val):
            return [(row['indicator'], val)]
        return []

    flat = []
    for _, row in domains_df.iterrows():
        flat.extend(flatten_domains(row))

    flat_df = pd.DataFrame(flat, columns=['indicator', 'domain'])

    # Count unique indicators per domain
    domain_counts = (
        flat_df.groupby('domain')['indicator']
        .nunique()
        .reset_index()
        .rename(columns={'indicator': 'indicator_count'})
        .sort_values('indicator_count', ascending=False)
    )

    display(domain_counts)
else:
    display("Column 'enrich_domains' does not exist in the provided DataFrame. Skipping domain association count.")

"Column 'enrich_domains' does not exist in the provided DataFrame. Skipping domain association count."

In [135]:
import os
import pandas as pd
from datetime import datetime, timedelta

# Base file path with placeholder for date
base_path = r"Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d{date}.csv"
#base_path = r"C:\Users\jaskew\Documents\project_repository\data\raw\ObservationDataFiles\htoc_opdiv_obs_d{date}.csv"
date_format = "%Y%m%d"

def get_file_paths(base_path, days):
    """ Generate file paths for the last `days` days using list comprehension. """
    today = datetime.utcnow()
    dates_to_pull = [(today - timedelta(days=i)).strftime(date_format) for i in range(days)]
    
    # Construct file paths
    file_paths = [base_path.format(date=dt) for dt in dates_to_pull]
    
    # Filter for existing files
    existing_files = [file_path for file_path in file_paths if os.path.exists(file_path)]
    
    if not existing_files:
        print("No files found for the specified date range.")
    else:
        print(f"Files to be loaded: {existing_files}")
    
    return existing_files

def load_observed_data(file_paths):
    """ Load and concatenate observed data from multiple files. """
    data_frames = []

    for file_path in file_paths:
        try:
            df = pd.read_csv(file_path)
            data_frames.append(df)
        except Exception as e:
            print(f"Error reading file {file_path}: {e}")
    
    # Concatenate data
    if data_frames:
        observed_data_df = pd.concat(data_frames, ignore_index=True)
        print(f"Loaded data from {len(data_frames)} files.")
    else:
        observed_data_df = pd.DataFrame()

    return observed_data_df

# Example Usage:
# Fetch file paths for the last 3 days
file_paths = get_file_paths(base_path, days=365)

# Load observed data
observed_data_df = load_observed_data(file_paths)



C:\Users\jaskew\AppData\Local\Temp\ipykernel_19924\1921769075.py:12: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  today = datetime.utcnow()


Files to be loaded: ['Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20251110.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20251109.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20251108.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20251107.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20251106.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20251105.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20251104.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20251103.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20251102.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20251101.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20251031.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20251030.csv', 'Z:/HTOC/Data_Analytics/Data/Op

In [136]:
# Aggregate observed_data_df by 'indicator', counting unique obs_date occurrences
agg_by_indicator = (
    observed_data_df
    .groupby('indicator', as_index=False)['obs_date']
    .nunique()
    .rename(columns={'obs_date': 'obs_days_count'})
)

agg_by_indicator = agg_by_indicator[agg_by_indicator['indicator'].isin(recent_tags['indicator'])]
# Map obs_days_count from agg_by_indicator to recent_tags by indicator, rename to obs_count
recent_tags = recent_tags.merge(
    agg_by_indicator.rename(columns={'obs_days_count': 'obs_count'}),
    on='indicator',
    how='left'
)
display(agg_by_indicator)

,indicator,obs_days_count
19,101.71.130.99,26
21,101.89.174.236,174
94,103.133.60.12,48
109,103.150.218.78,26
111,103.153.149.26,12
...,...,...
5615,www.deepseek.com,267
5616,www.deepseek.com.cdn.cloudflare.net,216
5649,www.shorturl.at/,174
5652,www.sthda.com,248


In [137]:
import re

# helper to flatten list-or-str into a list of strings
def flatten_hosts(hosts):
    if isinstance(hosts, list):
        return [h for h in hosts if isinstance(h, str)]
    elif isinstance(hosts, str):
        return [hosts]
    return []

# regex: 
# ^[A-Za-z]+       → first label is one “word” (letters only)
# (?:\.[^.]+){3}$  → exactly three more “.” followed by non-dot characters
four_label_word_prefix = re.compile(r'^[A-Za-z]+(?:\.[^.]+){3}$')

# Try to match on 'enrich_org' instead of 'enrich_hostNames'
if 'enrich_org' in recent_tags.columns:
    global matched_results
    exploded = recent_tags.explode('enrich_org')

    mask = exploded['enrich_org'].astype(str).str.match(four_label_word_prefix, na=False)

    matched = recent_tags.loc[exploded[mask].index.unique()]
    matched_results = matched[['indicator','enrich_org']]
else:
    matched = pd.DataFrame(columns=recent_tags.columns)
    matched_results = None


display(matched_results)


,indicator,enrich_org


In [138]:
def has_false_positive(tag_str):
    if not isinstance(tag_str, str):
        return False
    tags = [t.strip().lower() for t in tag_str.split(",")]
    return any("false positive" in t for t in tags)

false_positive_indicators = recent_tags[
    recent_tags['tag_name'].apply(has_false_positive)
]

display(false_positive_indicators[['indicator', 'tag_name']])


,indicator,tag_name


In [139]:
def has_false_positive(tag_str):
    if not isinstance(tag_str, str):
        return False
    tags = [t.strip().lower() for t in tag_str.split(",")]
    return any("false positive" in t for t in tags)

false_positive_indicators = recent_tags[
    recent_tags['tag_name'].apply(has_false_positive)
]

display(false_positive_indicators[['indicator', 'tag_name']])


,indicator,tag_name


In [152]:
import pandas as pd
import numpy as np

df = recent_tags.copy()

# ── Severity Binning ────────────────────────────
scoring_bins = [0, 200, 500, 800, 1001]
label_bins = ['low', 'medium', 'high', 'critical']

# ── Feature Weights / Params ────────────────────
Weights = {
    "MALICIOUS_WEIGHT": 1.00,
    "OBSERVATION_COUNT_WEIGHT": 0.02,
    "CONTINUITY_WEIGHT": 0.10,
    "TC_RATING": 0.05,
    "TC_CONFIDENCE": 0.02,
    "TOR_ACTIVITY_WEIGHT": 3.00,  # Bonus for TOR activity
}

BOTNET_ACTIONS = {
    'scanning','ddos','spam','phishing','cryptojacking','credential stuffing','ransomware'
}

TOR_ACTIVITY = {
    'tor','tor activity'
}

# ── Utility Functions ────────────────────────────────────────────────
def convert_to_list(val):
    """
    Convert various input types to a list format.
    Handles: None, NaN, lists, tuples, sets, strings (including list representations and comma-separated).
    """
    if val is None or (isinstance(val, float) and pd.isna(val)):
        return []
    if isinstance(val, (list, set, tuple)):
        return list(val)
    if isinstance(val, str):
        # Try to parse as list if it looks like one
        if val.strip().startswith('[') and val.strip().endswith(']'):
            try:
                import ast
                parsed = ast.literal_eval(val)
                if isinstance(parsed, (list, tuple)):
                    return list(parsed)
            except:
                pass
        # Otherwise treat as comma-separated string
        return [x.strip() for x in val.split(',') if x.strip()]
    return [val] if val else []

# Known feature maximums (absolute, domain-informed)
VT_MAX = 94
MAX_OBS_REALISTIC = 365
MAX_RATING = 5
MAX_CONFIDENCE = 100

# Policy multipliers
FALSE_POSITIVE_WEIGHT = 0.9         # 10% penalty
BOTNET_MULTIPLIER     = 0.4         # 60% penalty when botnet-related
SCANNER_PENALTY_MULTIPLIER = 0.7    # 30% penalty for scanner/benign-crawler tags
DATA_QUALITY_FLOOR    = 0.85        # at worst 15% reduction for poor completeness

# ── Input caps ──────────────────────────────────
df['obs_count'] = pd.to_numeric(df.get('obs_count', 0), errors='coerce').fillna(0).clip(0, MAX_OBS_REALISTIC)
df['enrich_vtMaliciousCount'] = (
    pd.to_numeric(df.get('enrich_vtMaliciousCount', 0), errors='coerce')
      .fillna(0).clip(0, VT_MAX)
)
df['rating']     = pd.to_numeric(df.get('rating', 0), errors='coerce').fillna(0).clip(0, MAX_RATING)
df['confidence'] = pd.to_numeric(df.get('confidence', 0), errors='coerce').fillna(0).clip(0, MAX_CONFIDENCE)
df['type'] = df.get('type', '').astype(str)

# ── Base additive evidence (no obs additive term) ───────────────
df['malicious_scaled']     = np.power(df['enrich_vtMaliciousCount'], 0.4)
df['malicious_raw_score']  = df['malicious_scaled'] * Weights['MALICIOUS_WEIGHT']
df['continuity_val']       = df['type'].map({
    'Address': 1, 'IPv4': 1, 'IPv6': 1,
    'Domain': 2, 'Host': 2, 'URL': 2, 'Stripped URL': 2,
    'EmailAddress': 2, 'EmailSubject': 2,
    'SHA1': 3, 'SHA256': 3, 'MD5': 3
}).fillna(0)
df['continuity_raw_score'] = df['continuity_val'] * Weights['CONTINUITY_WEIGHT']
df['tc_raw_rating']        = df['rating']     * Weights['TC_RATING']
df['tc_raw_confidence']    = df['confidence'] * Weights['TC_CONFIDENCE']

# ── TOR Activity detection (check enrich_tags, tag_name, and enrich columns) ───────────────
def has_tor_activity(val):
    # Use the existing TOR_ACTIVITY set for case-insensitive matching
    # Handle None, NaN, or empty values
    if val is None or (isinstance(val, float) and pd.isna(val)):
        return False
    
    # Handle list, set, or tuple
    if isinstance(val, (list, set, tuple)):
        if len(val) == 0:
            return False
        t = " ".join(map(str, val)).lower()
    # Handle string (supports csv-ish strings)
    elif isinstance(val, str):
        if not val or val.strip() == '':
            return False
        t = " ".join(x.strip() for x in val.split(',')).lower()
    else:
        t = str(val).lower()
        if t in ['nan', 'none', '']:
            return False
    
    return any(keyword in t for keyword in TOR_ACTIVITY)

# Check for TOR activity in tag_name and enrich_tags only
tor_mask_tag = pd.Series(False, index=df.index)
tor_mask_enrich_tags = pd.Series(False, index=df.index)

if 'enrich_tags' in df.columns:
    tor_mask_enrich_tags = df['enrich_tags'].apply(has_tor_activity)

if 'tag_name' in df.columns:
    # Convert tag_name to list using global convert_to_list function
    tag_name_as_list = df['tag_name'].apply(convert_to_list)
    tor_mask_tag = tag_name_as_list.apply(has_tor_activity)

# Combine TOR masks (OR logic) - only checking tag_name and enrich_tags
tor_flag = (tor_mask_enrich_tags | tor_mask_tag).astype(int)
df['tor_activity_score'] = tor_flag * Weights['TOR_ACTIVITY_WEIGHT']

df['raw_score'] = (
    df['malicious_raw_score'] +
    df['continuity_raw_score'] +
    df['tc_raw_rating'] +
    df['tc_raw_confidence'] +
    df['tor_activity_score']
)

# ── Observation penalty (multiplier; linear) ────────────────────
OBS_PENALTY_STRENGTH = float(Weights['OBSERVATION_COUNT_WEIGHT'])
OBS_MIN_MULTIPLIER   = 0.50  # floor so max reduction is 50% (tune to taste)

obs_frac = df['obs_count'] / MAX_OBS_REALISTIC
df['obs_penalty_multiplier'] = (1.0 - OBS_PENALTY_STRENGTH * obs_frac).clip(OBS_MIN_MULTIPLIER, 1.0)
df['raw_score'] *= df['obs_penalty_multiplier']

# ── Data quality multiplier (light guard) ───────────────────────
needed = ['type','rating','confidence','enrich_vtMaliciousCount']
present_frac = df[needed].notna().sum(axis=1) / len(needed)
df['data_quality_multiplier'] = present_frac.clip(DATA_QUALITY_FLOOR, 1.0)
df['raw_score'] *= df['data_quality_multiplier']

# ── Botnet flag (robust, multi-word, list/string) ───────────────
col = df.get('Botnet', None)
if col is None:
    df['botnet_flag'] = 0
else:
    def is_botnet(val):
        text = " ".join(map(str, val)).lower() if isinstance(val, (list, set, tuple)) else str(val).lower()
        return int(any(action in text for action in BOTNET_ACTIONS))
    df['botnet_flag'] = pd.Series(col).apply(is_botnet).astype(int)

# ── False Positive penalty (preview + conditional apply) ────────
if 'falsePositives' in df.columns:
    df['falsePositives'] = pd.to_numeric(df['falsePositives'], errors='coerce').fillna(0)
    mask_fp = df['falsePositives'] > 0
    # Preview: always "what if" penalized
    df['false_positive_raw_score'] = df['raw_score'] * FALSE_POSITIVE_WEIGHT
    # Apply FP only where present
    df.loc[mask_fp, 'raw_score'] = df.loc[mask_fp, 'false_positive_raw_score']
else:
    df['false_positive_raw_score'] = df['raw_score']

# ── Botnet penalty application ──────────────────────────────────
df['botnet_penalty_multiplier'] = np.where(df['botnet_flag'] == 1, BOTNET_MULTIPLIER, 1.0)
df['raw_score'] *= df['botnet_penalty_multiplier']

# ── Scanner penalty (robust tag parse) ──────────────────────────
def has_scanner_tag(val):
    # Standardize all scanner tags to lowercase for case-insensitive matching
    scanners = {'scanner', 'masscan', 'zmap', 'shodan', 'censys', 'active scanning: scanning ip blocks', 'web scanner', 'active scanning'}
    
    # Handle None, NaN, or empty values
    if val is None or (isinstance(val, float) and pd.isna(val)):
        return False
    
    # Handle list, set, or tuple
    if isinstance(val, (list, set, tuple)):
        if len(val) == 0:
            return False
        t = " ".join(map(str, val)).lower()
    # Handle string (supports csv-ish strings)
    elif isinstance(val, str):
        if not val or val.strip() == '':
            return False
        t = " ".join(x.strip() for x in val.split(',')).lower()
    else:
        t = str(val).lower()
        if t in ['nan', 'none', '']:
            return False
    
    return any(s in t for s in scanners)

# Check both enrich_tags and tag_name columns
scanner_mask_enrich = pd.Series(False, index=df.index)
scanner_mask_tag = pd.Series(False, index=df.index)

if 'enrich_tags' in df.columns:
    scanner_mask_enrich = df['enrich_tags'].apply(has_scanner_tag)

if 'tag_name' in df.columns:
    # Convert tag_name to list using global convert_to_list function
    tag_name_as_list = df['tag_name'].apply(convert_to_list)
    scanner_mask_tag = tag_name_as_list.apply(has_scanner_tag)

# Combine both masks (OR logic - if either has scanner tag, apply penalty)
scanner_mask = scanner_mask_enrich | scanner_mask_tag

df['scanner_penalty_multiplier'] = np.where(scanner_mask, SCANNER_PENALTY_MULTIPLIER, 1.0)
df['raw_score'] *= df['scanner_penalty_multiplier']

# ── Absolute Cap (multipliers are NOT in cap; only additive parts) ───────────
RAW_SCORE_CAP = (
    np.log1p(VT_MAX) * Weights['MALICIOUS_WEIGHT'] +
    float(df['continuity_val'].max() if len(df) else 3) * Weights['CONTINUITY_WEIGHT'] +
    (MAX_RATING * Weights['TC_RATING']) +
    (MAX_CONFIDENCE * Weights['TC_CONFIDENCE']) +
    Weights['TOR_ACTIVITY_WEIGHT']  # TOR activity bonus
)

# ── Normalize to 0–1000 ─────────────────────────────────────────
df['Threat_Score'] = (1000 * (df['raw_score'] / RAW_SCORE_CAP).clip(0, 1)).round().fillna(0).astype(int)

# ── VT-driven score ceilings / floors ───────────────────────────
if 'enrich_vtMaliciousCount' in df.columns:
    vt_counts = pd.to_numeric(df['enrich_vtMaliciousCount'], errors='coerce').fillna(0)
    low_cap_mask = vt_counts <= 4
    medium_cap_mask = vt_counts.between(5, 10, inclusive='both')
    high_floor_mask = vt_counts >= 13

    # Ceiling for low VT: keep within low severity band
    low_max_score = scoring_bins[1] - 1  # top of "low" bin (e.g., 199)
    df.loc[low_cap_mask, 'Threat_Score'] = df.loc[low_cap_mask, 'Threat_Score'].clip(upper=low_max_score)

    # Ceiling for medium VT: keep within medium severity band
    medium_max_score = scoring_bins[2] - 1  # top of "medium" bin (e.g., 499)
    df.loc[medium_cap_mask, 'Threat_Score'] = df.loc[medium_cap_mask, 'Threat_Score'].clip(upper=medium_max_score)

    # Floor for high VT: ensure at least maximum medium score
    df.loc[high_floor_mask, 'Threat_Score'] = df.loc[high_floor_mask, 'Threat_Score'].clip(lower=medium_max_score)

# Cap botnet-flagged indicators to low severity ceiling
if 'botnet_flag' in df.columns:
    low_max_score = scoring_bins[1] - 1
    df.loc[df['botnet_flag'] == 1, 'Threat_Score'] = df.loc[df['botnet_flag'] == 1, 'Threat_Score'].clip(upper=low_max_score)

# Ensure Threat_Score remains integer after clipping
df['Threat_Score'] = df['Threat_Score'].round().astype(int)

# ── Assign Severity bin ─────────────────────────────────────────
df['Severity'] = pd.cut(df['Threat_Score'], bins=scoring_bins, labels=label_bins, right=False)

# ── Explanation (drivers + multipliers) ─────────────────────────
_NAME_MAP = {
    'malicious_raw_score': 'VT malicious (log-scaled)',
    'continuity_raw_score': 'Continuity (indicator type)',
    'tc_raw_rating': 'TC rating',
    'tc_raw_confidence': 'TC confidence',
    'tor_activity_score': 'TOR activity',
}

def build_explanation(row):
    parts = {
        'malicious_raw_score': float(row.get('malicious_raw_score', 0) or 0),
        'continuity_raw_score': float(row.get('continuity_raw_score', 0) or 0),
        'tc_raw_rating': float(row.get('tc_raw_rating', 0) or 0),
        'tc_raw_confidence': float(row.get('tc_raw_confidence', 0) or 0),
        'tor_activity_score': float(row.get('tor_activity_score', 0) or 0),
    }
    score = float(row.get('Threat_Score', 0) or 0)
    sev = str(row.get('Severity', 'nan'))

    # Top additive drivers (without calculations)
    contrib = sorted(parts.items(), key=lambda kv: abs(kv[1]), reverse=True)[:3]
    driver_bits = []
    for k, v in contrib:
        if v != 0:  # Only include non-zero drivers
            label = _NAME_MAP.get(k, k)
            driver_bits.append(label)

    # Multipliers
    botnet_mult  = float(row.get('botnet_penalty_multiplier', 1.0))
    scanner_mult = float(row.get('scanner_penalty_multiplier', 1.0))
    fp_cnt       = int(row.get('falsePositives', 0) or 0)
    tor_score    = float(row.get('tor_activity_score', 0) or 0)

    botnet_note = "Botnet penalty applied." if botnet_mult < 1.0 else "No botnet penalty."
    fp_note     = f"{fp_cnt} false positive tag(s)." if fp_cnt > 0 else "No false positive tags."
    scan_note   = "Scanner penalty applied." if scanner_mult < 1.0 else "No scanner penalty."
    tor_note    = "TOR activity detected." if tor_score > 0 else "No TOR activity."

    drivers_text = "; ".join(driver_bits) if driver_bits else "No significant drivers"

    return (
        f"Severity: {sev}. Top drivers: {drivers_text}. "
        f"{botnet_note} {fp_note} {scan_note} {tor_note} Score: {score:.0f}/1000."
    )

df['Explanation'] = df.apply(build_explanation, axis=1)

# ── De-dupe and show ────────────────────────────────────────────
if 'indicator' in df.columns:
    df.drop_duplicates(subset='indicator', inplace=True)

display(df[['indicator','type','enrich_vtMaliciousCount','obs_count',
            'rating','obs_penalty_multiplier',
            'botnet_flag','falsePositives',
            'Threat_Score','Severity','Explanation']])


,indicator,type,enrich_vtMaliciousCount,obs_count,rating,obs_penalty_multiplier,botnet_flag,falsePositives,Threat_Score,Severity,Explanation
0,101.71.130.99,Address,10.0,26,3.0,0.998575,0,0,429,medium,Severity: medium. Top drivers: VT malicious (l...
1,101.89.174.236,Address,9.0,174,3.0,0.990466,0,0,376,medium,Severity: medium. Top drivers: VT malicious (l...
2,103.133.60.12,Address,1.0,48,1.0,0.997370,1,0,199,low,Severity: low. Top drivers: TOR activity; TC c...
3,103.150.218.78,Address,1.0,26,2.0,0.998575,1,0,199,low,Severity: low. Top drivers: TOR activity; TC c...
4,103.153.149.26,Address,0.0,12,1.0,0.999342,1,0,171,low,Severity: low. Top drivers: TOR activity; TC c...
...,...,...,...,...,...,...,...,...,...,...,...
834,www.deepseek.com,Host,0.0,267,3.0,0.985370,0,0,172,low,Severity: low. Top drivers: TC confidence; Con...
835,www.deepseek.com.cdn.cloudflare.net,Host,0.0,216,3.0,0.988164,0,0,169,low,Severity: low. Top drivers: TC confidence; Con...
836,www.shorturl.at/,Stripped URL,0.0,174,3.0,0.990466,0,0,199,low,Severity: low. Top drivers: TC confidence; Con...
837,www.sthda.com,Host,0.0,248,4.0,0.986411,0,0,110,low,Severity: low. Top drivers: TC confidence; Con...


In [141]:
df[df['indicator'] == '190.92.174.36']

,indicator,dateAdded,ownerId,ownerName,webLink,type,lastModified,falsePositives,rating,confidence,...,raw_score,obs_penalty_multiplier,data_quality_multiplier,botnet_flag,false_positive_raw_score,botnet_penalty_multiplier,scanner_penalty_multiplier,Threat_Score,Severity,Explanation
161,190.92.174.36,2025-04-11T17:46:48Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-11-10T08:40:56Z,0,4.0,63,...,2.542045,0.992986,1.0,0,2.28784,1.0,1.0,254,medium,Severity: medium. Top drivers: TC confidence; ...


In [157]:
# Save the DataFrame to Excel with only the columns displayed in cell 14
import os
import pandas as pd
from datetime import datetime
from openpyxl.styles import PatternFill

output_dir = r"Z:\HTOC\Data_Analytics\Data\Threat Assessment Scores"
os.makedirs(output_dir, exist_ok=True)

# Create filename
excel_filename = "Threat_Assessment_Scores.xlsx"
excel_path = os.path.join(output_dir, excel_filename)

# Select only the columns displayed in cell 14
columns_to_save = ['indicator', 'type', 'enrich_vtMaliciousCount', 'obs_count',
                   'rating', 'obs_penalty_multiplier',
                   'botnet_flag', 'falsePositives',
                   'Threat_Score', 'Severity', 'Explanation']

# Make a copy with only selected columns
df_export = df[columns_to_save].copy()

# Convert timezone-aware datetime columns to timezone-naive for Excel compatibility
for col in df_export.columns:
    if pd.api.types.is_datetime64_any_dtype(df_export[col]):
        # Check if the column has timezone info
        if df_export[col].dt.tz is not None:
            # Convert to UTC first, then remove timezone info to make it timezone-naive
            df_export[col] = df_export[col].dt.tz_convert('UTC').dt.tz_localize(None)

# Check if file exists and read existing data
if os.path.exists(excel_path):
    df_existing = pd.read_excel(excel_path, engine='openpyxl')
    
    # Ensure both dataframes have the same columns in the same order
    df_existing = df_existing[columns_to_save].copy()
    
    # Count how many indicators will be updated vs added
    existing_indicators = set(df_existing['indicator'].values)
    new_indicators = set(df_export['indicator'].values)
    
    indicators_to_add = len(new_indicators - existing_indicators)
    indicators_to_check = existing_indicators & new_indicators
    
    # Find indicators that have actually changed
    indicators_to_update = []
    indicators_unchanged = []
    
    # Set indicator as index for comparison
    df_existing_idx = df_existing.set_index('indicator').sort_index()
    df_export_idx = df_export.set_index('indicator').sort_index()
    
    for indicator in indicators_to_check:
        # Compare the rows (excluding the index/indicator column)
        existing_row = df_existing_idx.loc[indicator]
        new_row = df_export_idx.loc[indicator]
        
        # Check if any values have changed
        if not existing_row.equals(new_row):
            indicators_to_update.append(indicator)
        else:
            indicators_unchanged.append(indicator)
    
    # Build the combined dataframe: keep existing unchanged records, update changed ones, add new ones
    # Start with existing records that are unchanged
    df_unchanged = df_existing[df_existing['indicator'].isin(indicators_unchanged)].copy()
    
    # Add updated records (from new data)
    df_updated = df_export[df_export['indicator'].isin(indicators_to_update)].copy()
    
    # Add new records (not in existing)
    df_new = df_export[df_export['indicator'].isin(new_indicators - existing_indicators)].copy()
    
    # Add existing records that are not in new data (preserve historical data)
    df_preserved = df_existing[~df_existing['indicator'].isin(new_indicators)].copy()
    
    # Combine all parts
    df_combined = pd.concat([df_unchanged, df_updated, df_new, df_preserved], ignore_index=True)
    
    # Final check: remove any duplicates (shouldn't happen, but safety check)
    df_combined = df_combined.drop_duplicates(subset='indicator', keep='last')
    
    print(f"Checked {len(indicators_to_check)} existing indicators")
    print(f"  - Updated {len(indicators_to_update)} indicators with changes")
    print(f"  - Kept {len(indicators_unchanged)} indicators unchanged")
    print(f"Added {indicators_to_add} new indicators")
    print(f"Total unique indicators in file: {len(df_combined)}")
else:
    df_combined = df_export.copy()
    # Remove any duplicates in the new data itself
    df_combined = df_combined.drop_duplicates(subset='indicator', keep='last')
    print(f"Created new file with {len(df_combined)} indicators")

# Save to Excel with severity-based highlighting
with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
    df_combined.to_excel(writer, index=False, sheet_name='Threat Scores')

    workbook = writer.book
    worksheet = writer.sheets['Threat Scores']

    # Define fills for each severity
    fills = {
        'low': PatternFill(start_color='83de85', end_color='83de85', fill_type='solid'),     # light green
        'medium': PatternFill(start_color='eef084', end_color='eef084', fill_type='solid'),  # light yellow
        'high': PatternFill(start_color='f29953', end_color='f29953', fill_type='solid'),    # light orange
        'critical': PatternFill(start_color='e83f3f', end_color='e83f3f', fill_type='solid') # light red
    }

    for row_idx, severity in enumerate(df_combined['Severity'], start=2):  # start=2 to skip header
        fill = fills.get(str(severity).lower())
        if fill:
            for col_idx in range(1, len(df_combined.columns) + 1):
                worksheet.cell(row=row_idx, column=col_idx).fill = fill

print(f"Saved {len(df_combined)} total indicators with threat assessment scores to {excel_path}")


Checked 839 existing indicators
  - Updated 0 indicators with changes
  - Kept 839 indicators unchanged
Added 0 new indicators
Total unique indicators in file: 1068
Saved 1068 total indicators with threat assessment scores to Z:\HTOC\Data_Analytics\Data\Threat Assessment Scores\Threat_Assessment_Scores.xlsx
